<a href="https://colab.research.google.com/github/DongAnYu/ai-tutor-rag-system/blob/main/notebooks/07-RAG_Improve_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Packages and Setup Variables


In [2]:
!pip install -q llama-index==0.14.0 openai==1.107.0 chromadb==1.0.21 llama-index-vector-stores-chroma==0.5.3 \
                llama-index-llms-google-genai==0.5.0 cohere==5.18.0 jedi==0.19.2

In [3]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"


from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')

In [4]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

# Load a Model


In [5]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})

In [ ]:
# Gemini (Free Tier) - RPM Quota error might happen.

# from llama_index.llms.google_genai import GoogleGenAI
# llm = GoogleGenAI(model="gemini-2.5-flash", temperature=0.3)

# Create a VectoreStore


In [ ]:
import chromadb

# create client and a new collection
# chromadb.EphemeralClient saves data in-memory.
chroma_client = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = chroma_client.create_collection("mini-llama-articles")

In [9]:
from llama_index.vector_stores.chroma import ChromaVectorStore

# Define a storage context object using the created vector database.
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Load the Dataset (CSV)


## Download


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model. Read the dataset as a long string.


In [10]:
!curl -o ./mini-llama-articles.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  169k  100  169k    0     0   793k      0 --:--:-- --:--:-- --:--:--  792k


## Read File


In [11]:
import csv

rows = []

# Load the file as a JSON
with open("./mini-llama-articles.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        rows.append(row)

# The number of characters in the dataset.
len(rows)

14

# Convert to Document obj


In [12]:
from llama_index.core import Document

# Convert the chunks to Document objects so the LlamaIndex framework can process them.
documents = [
    Document(
        text=row[1], metadata={"title": row[0], "url": row[2], "source_name": row[3]}
    )
    for row in rows
]

# Transforming


In [13]:
from llama_index.core.node_parser import TokenTextSplitter

# Define the splitter object that split the text into segments with 512 tokens,
# with a 128 overlap between the segments.
text_splitter = TokenTextSplitter(separator=" ", chunk_size=512, chunk_overlap=128)

In [14]:
from llama_index.core.extractors import (
    SummaryExtractor,
    QuestionsAnsweredExtractor,
    KeywordExtractor,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

# Create the pipeline to apply the transformation on each chunk,
# and store the transformed text in the chroma vector store.
pipeline = IngestionPipeline(
    transformations=[
        text_splitter,   # Splits large documents into smaller chunks
        QuestionsAnsweredExtractor(questions=3, llm=llm),  # For each chunk, the LLM generates 3 questions that this chunk can answer.
        SummaryExtractor(summaries=["prev", "self"], llm=llm), # Generates summaries: "self" → summary of this chunk, summary of the previous chunk
        KeywordExtractor(keywords=10, llm=llm),  # Extracts 10 important keywords per chunk.
        OpenAIEmbedding(),  # Turns the final enriched text into a numerical vector:
    ],
    vector_store=vector_store,
)

# Run the transformation pipeline.
nodes = pipeline.run(documents=documents, show_progress=True)

Parsing nodes:   0%|          | 0/14 [00:00<?, ?it/s]

100%|██████████| 108/108 [00:38<00:00,  2.80it/s]


Generating embeddings:   0%|          | 0/108 [00:00<?, ?it/s]

In [38]:
node = nodes[0]
print(node.metadata)

{'title': "Beyond GPT-4: What's New?", 'url': 'https://pub.towardsai.net/beyond-gpt-4-whats-new-cbd61a448eb9#dda8', 'source_name': 'towards_ai', 'questions_this_excerpt_can_answer': '1. According to the article, what are the three specialized variants of Meta’s Code Llama and what are each of their intended uses?  \n\n2. How does the article characterize Llama 2-Chat’s performance in human-centric evaluations compared to proprietary closed-source models?  \n\n3. Which publicly available model did Code Llama reportedly not outperform on code benchmarks, based on the article’s claims?', 'section_summary': "Summary:\n- Main topics:\n  - Meta's Llama 2 family and Code Llama release and capabilities\n  - Model variants, fine-tuning, and benchmarking results\n  - Comparison between open-source models and proprietary models (notably GPT-4)\n  - The shift from text-only LLMs to multimodal models\n\n- Key entities:\n  - Meta (developer of Llama 2 and Code Llama)\n  - Llama 2 (7B–70B parameter m

In [15]:
len(nodes)

108

In [16]:
# Compress the vector store directory to a zip file to be able to download and use later.
!zip -r vectorstore.zip mini-llama-articles

  adding: mini-llama-articles/ (stored 0%)
  adding: mini-llama-articles/chroma.sqlite3 (deflated 67%)
  adding: mini-llama-articles/41b94133-1014-4f4f-a227-e49ab6a529d9/ (stored 0%)
  adding: mini-llama-articles/41b94133-1014-4f4f-a227-e49ab6a529d9/header.bin (deflated 63%)
  adding: mini-llama-articles/41b94133-1014-4f4f-a227-e49ab6a529d9/link_lists.bin (stored 0%)
  adding: mini-llama-articles/41b94133-1014-4f4f-a227-e49ab6a529d9/data_level0.bin (deflated 100%)
  adding: mini-llama-articles/41b94133-1014-4f4f-a227-e49ab6a529d9/length.bin (deflated 33%)


# Load Indexes


If you have already uploaded the zip file for the vector store checkpoint, please uncomment the code in the following cell block to extract its contents. After doing so, you will be able to load the dataset from local storage.


In [17]:
# !unzip vectorstore.zip

Archive:  vectorstore.zip
replace mini-llama-articles/chroma.sqlite3? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [18]:
# Load the vector store from the local storage.
db = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = db.get_or_create_collection("mini-llama-articles")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [19]:
from llama_index.core import VectorStoreIndex

# Create the index based on the vector store.
index = VectorStoreIndex.from_vector_store(vector_store)

# Query Dataset


In [20]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

res = query_engine.query("How many parameters LLaMA2 model has?")

In [21]:
res.response

'LLaMA2 model variants include 7 billion, 13 billion, 34 billion, and 70 billion parameters.'

In [22]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 c8be0698-a158-4209-a587-84932eb6852e
Title	 The Generative AI Revolution: Exploring the Current Landscape
Text	 Models Meta AI, formerly known as Facebook Artificial Intelligence Research (FAIR), is an artificial intelligence laboratory that aims to share open-source frameworks, tools, libraries, and models for research exploration and large-scale production deployment. In 2018, they released the open-source PyText, a modeling framework focused on NLP systems. Then, in August 2022, they announced the release of BlenderBot 3, a chatbot designed to improve conversational skills and safety. In November 2022, Meta developed a large language model called Galactica, which assists scientists with tasks such as summarizing academic papers and annotating molecules and proteins. Released in February 2023, LLaMA (Large Language Model Meta AI) is a transformer-based foundational large language model by Meta that ventures into both the AI and academic spaces. The model aims to help researc

### Trying a different Query


In [23]:
res = query_engine.query("Does GQA helped LLaMA performance?")

In [24]:
res.response

'The excerpt does not mention GQA or any effect of GQA on Llama/Llama2 performance. Therefore there is no information provided that GQA helped LLaMA’s performance.'

In [25]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 a16bd42c-5be8-4813-8dbd-df813772823b
Title	 Exploring Large Language Models -Part 3
Text	 a commodity GPU and then fine-tune them. There are two parts to this- Quantisation and Parameter Efficient Tuning. The real magic of this is that a laptop with a sufficient recent GPU (having Tensor Cores), can run the 7 billion Lamma2 pre-trained model open-sourced recently by Meta Research. Imagine the compressed knowledge and an NLU (Natural Language Understanding) model running on your local laptop. This is still a smallish model, but it's still capable of understanding and has sufficient world knowledge embedded in it to be quite useful. Imagine what a model like this or better models in the future could do if it could run in small servers or in cars, and leverage its causal reasoning and world model knowledge to supervise lower-level/specialist AI/ML systems. So we have now a way to fit reasonably large models (7B or more) in a single GPU, via Quantisation and then train them in a p

From the articles:

> [...]The 7 billion model of Llama2 has sufficient NLU (Natural Language Understanding) to create output based on a particular format[...]


# No Metadata


Now, let's evaluate the ability of the query engine independently of the generated metadata, like keyword extraction or summarization.

How good is retrieval if I use vectors only, with no LLM-generated metadata?



In [26]:
from llama_index.core import Document

documents_no_meta = [Document(text=row[1]) for row in rows]

In [27]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        OpenAIEmbedding(),
    ]
)

nodes_no_meta = pipeline.run(documents=documents_no_meta, show_progress=True)

Parsing nodes:   0%|          | 0/14 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/94 [00:00<?, ?it/s]

In [28]:
index_no_metadata = VectorStoreIndex(nodes=nodes_no_meta)

query_engine_no_metadata = index_no_metadata.as_query_engine(llm=llm)

In [29]:
res = query_engine_no_metadata.query("Does GQA helped LLaMA performance?")

In [30]:
res.response

'The provided information does not mention GQA or describe any use of GQA in training or improving LLaMA. Therefore, based on the given material, there is no evidence that GQA helped LLaMA performance.'

In [31]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 f0355f93-3ae1-46c7-b384-67ab20675373
Text	 137B parameters and perform instruction tuning ... Since we use QLoRa we are effectively closely following this paper - QLORA: Efficient Finetuning of Quantized LLMs concerning the training data set, the format that the authors used to train their Gauanco model This is the format for the Llama2 model and will be different for others. One of the hardest problems of training is finding or creating a good quality data set to train. In our case, converting the available training data set to the instruction data set. Since our use case is Closed Book QA, we need to convert this to a QA format. Using older NLP methods like NER (Named Entity Recognition) and then using that to create a QA dataset was not effective. This is where the Self-instruct concept could be used However previous to Llama2, the best-performing model was the GPT 3/4 model via ChatGPT or its API and using these models to do the same was expensive. The 7 billion model of L

# Evaluate


In [32]:
from llama_index.core.evaluation import generate_question_context_pairs

# Create questions for each segment. These questions will be used to
# assess whether the retriever can accurately identify and return the
# corresponding segment when queried.

rag_eval_dataset = generate_question_context_pairs(
    nodes, llm=llm, num_questions_per_chunk=1
)

# We can save the evaluation dataset as a json file for later use.
rag_eval_dataset.save_json("./rag_eval_dataset.json")

100%|██████████| 108/108 [03:11<00:00,  1.77s/it]


If you have uploaded the generated question JSON file, please uncomment the code in the next cell block. This will avoid the need to generate the questions manually, saving you time and effort.


In [ ]:
# from llama_index.finetuning.embeddings.common import (
#     EmbeddingQAFinetuneDataset,
# )
# rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json(
#     "./rag_eval_dataset.json"
# )

In [35]:
import pandas as pd


#  A simple function to show the evaluation result.
def display_results_retriever(name, eval_results):
    """Display results from evaluate."""

    metric_dicts = []
    for eval_result in eval_results:
        metric_dict = eval_result.metric_vals_dict
        metric_dicts.append(metric_dict)

    full_df = pd.DataFrame(metric_dicts)

    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()

    metric_df = pd.DataFrame(
        {"Retriever Name": [name], "Hit Rate": [hit_rate], "MRR": [mrr]}
    )

    return metric_df

Hit Rate → did the correct chunk appear at all?

MRR → how high did it rank?

In [36]:
from llama_index.core.evaluation import RetrieverEvaluator

# We can evaluate the retievers with different top_k values.
for i in [2, 4, 6, 8, 10]:
    retriever = index.as_retriever(similarity_top_k=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    print(display_results_retriever(f"Retriever top_{i}", eval_results))

    Retriever Name  Hit Rate       MRR
0  Retriever top_2   0.87037  0.763889
    Retriever Name  Hit Rate       MRR
0  Retriever top_4  0.953704  0.791667
    Retriever Name  Hit Rate      MRR
0  Retriever top_6  0.972222  0.79537
    Retriever Name  Hit Rate       MRR
0  Retriever top_8  0.981481  0.796693
     Retriever Name  Hit Rate       MRR
0  Retriever top_10  0.981481  0.796693


Faithfulness: Is the answer grounded in retrieved context?

Relevancy: Does the answer actually address the question?

In [37]:
from llama_index.core.evaluation import (
    RelevancyEvaluator,
    FaithfulnessEvaluator,
    BatchEvalRunner,
)
from llama_index.llms.openai import OpenAI

for i in [2, 4, 6, 8, 10]:
    # Set Faithfulness and Relevancy evaluators
    query_engine = index.as_query_engine(similarity_top_k=i, llm=llm)

    # While we use GPT 5 to answer questions, we can use GPT4 to evaluate the answers.
    llm_gpt5 = OpenAI(model="gpt-5", additional_kwargs={'reasoning_effort':'minimal'})

    faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_gpt5)
    relevancy_evaluator = RelevancyEvaluator(llm=llm_gpt5)

    # Run evaluation
    queries = list(rag_eval_dataset.queries.values())
    batch_eval_queries = queries[:20]

    runner = BatchEvalRunner(
        {"faithfulness": faithfulness_evaluator, "relevancy": relevancy_evaluator},
        workers=8,
    )
    eval_results = await runner.aevaluate_queries(
        query_engine, queries=batch_eval_queries
    )
    faithfulness_score = sum(
        result.passing for result in eval_results["faithfulness"]
    ) / len(eval_results["faithfulness"])
    print(f"top_{i} faithfulness_score: {faithfulness_score}")

    relevancy_score = sum(result.passing for result in eval_results["relevancy"]) / len(
        eval_results["relevancy"]
    )
    print(f"top_{i} relevancy_score: {relevancy_score}")
    print("-_" * 10)

top_2 faithfulness_score: 0.95
top_2 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_
top_4 faithfulness_score: 0.95
top_4 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_
top_6 faithfulness_score: 0.95
top_6 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_
top_8 faithfulness_score: 0.95
top_8 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_
top_10 faithfulness_score: 0.95
top_10 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_
